# Milestone 3 — Full Evaluation, Ablation Study and Error Analysis

Building on Milestones 1 and 2, this notebook runs the complete pipeline
on the full Spider dev set with every improvement incorporated:

Milestone 3 vs Milestone 2:
- Alias-aware schema validity checker (fixes T1/T2 wrong-table errors)
- Hybrid candidate picker (self-consistency among valid+executable candidates,
  falls back to reranker)
- 8 candidates per question (more diversity)
- max_new_tokens=256 + repetition_penalty (fixes incomplete/looping SQL)
- Full cross-milestone comparison table
- Detailed error categorisation and complexity analysis

## 1. Install Dependencies

Pin identical versions to all previous notebooks. The saved LoRA adapter
was created with these exact versions — any mismatch silently breaks loading.

In [ ]:
!pip uninstall -y trl transformers peft accelerate
!pip install -q transformers==4.47.1 peft==0.13.2 accelerate==1.2.1 \
    datasets sqlparse sentencepiece

Found existing installation: transformers 4.47.1
Uninstalling transformers-4.47.1:
  Successfully uninstalled transformers-4.47.1
Found existing installation: peft 0.13.2
Uninstalling peft-0.13.2:
  Successfully uninstalled peft-0.13.2
Found existing installation: accelerate 1.2.1
Uninstalling accelerate-1.2.1:
  Successfully uninstalled accelerate-1.2.1


## 2. Verify Environment

In [ ]:
import transformers, peft, torch
print("transformers:", transformers.__version__)
print("peft:        ", peft.__version__)
print("torch:       ", torch.__version__)
print("CUDA:        ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:         ", torch.cuda.get_device_name(0))

transformers: 4.47.1
peft:         0.13.2
torch:        2.10.0+cu128
CUDA:         True
GPU:          NVIDIA A100-SXM4-40GB


## 3. Mount Drive and Define All Paths

All inputs are loaded from Drive. All outputs are written to
MILESTONE3_DIR so they never overwrite Milestone 2 results.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, re, sqlite3, time
from collections import Counter
import torch
import pandas as pd

drive.mount('/content/drive')

DRIVE_ROOT     = Path("/content/drive/MyDrive/CS_512_Project")
MILESTONE1_DIR = DRIVE_ROOT / "milestone1_lora"
MILESTONE2_DIR = DRIVE_ROOT / "milestone2_schema_val"
MILESTONE3_DIR = DRIVE_ROOT / "milestone3_full_eval"
PROCESSED_DIR  = DRIVE_ROOT / "processed_data"
ADAPTER_DIR    = MILESTONE1_DIR / "final_adapter"
SPIDER_ROOT    = Path("/content/drive/MyDrive/CS_512_Project/spider/spider_data")
CSV_DIR        = MILESTONE3_DIR / "csv_outputs"

for p in [MILESTONE3_DIR, CSV_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Milestone 3 dir:", MILESTONE3_DIR)
print("Adapter dir:    ", ADAPTER_DIR)
print("Adapter exists: ", ADAPTER_DIR.exists())
print("Spider root:    ", SPIDER_ROOT)
print("Spider exists:  ", SPIDER_ROOT.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Milestone 3 dir: /content/drive/MyDrive/CS_512_Project/milestone3_full_eval
Adapter dir:     /content/drive/MyDrive/CS_512_Project/milestone1_lora/final_adapter
Adapter exists:  True
Spider root:     /content/drive/MyDrive/CS_512_Project/spider/spider_data
Spider exists:   True


## 4. Load Full Spider Dev Set

For Milestone 3 we use ALL available preprocessed dev examples.
Make sure you re-saved processed_dev.json from the Baseline notebook
with dev_spider (all 1034 examples) before running this cell.

In [ ]:
with open(PROCESSED_DIR / "processed_train.json", "r", encoding="utf-8") as f:
    processed_train = json.load(f)

with open(PROCESSED_DIR / "processed_dev.json", "r", encoding="utf-8") as f:
    processed_dev_raw = json.load(f)

print("processed_train:", len(processed_train))
print("processed_dev:  ", len(processed_dev_raw))
print("Sample keys:    ", list(processed_dev_raw[0].keys()))

processed_train: 2500
processed_dev:   1034
Sample keys:     ['db_id', 'question', 'prompt', 'target_sql', 'schema_text', 'sqlite_path', 'text']


## 5. Fix Stale SQLite Paths and Verify

The Baseline notebook baked in /content/text2sql_project/ paths which
do not exist here. Remap every example and assert at least one resolves.

In [ ]:
OLD_PREFIX = "/content/text2sql_project/spider_data"
NEW_PREFIX = str(SPIDER_ROOT)

def fix_sqlite_path(example):
    if example["sqlite_path"].startswith(OLD_PREFIX):
        example["sqlite_path"] = example["sqlite_path"].replace(
            OLD_PREFIX, NEW_PREFIX, 1
        )
    return example

processed_dev = [fix_sqlite_path(ex) for ex in processed_dev_raw]

sample_path = processed_dev[0]["sqlite_path"]
print("Remapped path:", sample_path)
print("File exists:  ", os.path.exists(sample_path))
assert os.path.exists(sample_path), (
    f"SQLite path still wrong — check NEW_PREFIX. Got: {sample_path}"
)
print(f"Total dev examples available: {len(processed_dev)}")

Remapped path: /content/drive/MyDrive/CS_512_Project/spider/spider_data/database/concert_singer/concert_singer.sqlite
File exists:   True
Total dev examples available: 1034


## 6. Load Spider Schema Map

Build table/column lookup sets used by every schema validity function.
Also pre-build column-table pair sets for alias-aware validation.

In [ ]:
with open(SPIDER_ROOT / "tables.json", "r", encoding="utf-8") as f:
    tables_data = json.load(f)

schema_map = {}
for db in tables_data:
    schema_map[db["db_id"]] = {
        "db_id":                 db["db_id"],
        "table_names_original":  db["table_names_original"],
        "column_names_original": db["column_names_original"],
        "column_types":          db["column_types"],
        "primary_keys":          db["primary_keys"],
        "foreign_keys":          db["foreign_keys"],
    }

# Fast lookup sets — built once, used everywhere
table_name_sets  = {}
column_name_sets = {}

for db_id, schema in schema_map.items():
    table_name_sets[db_id] = {
        t.lower() for t in schema["table_names_original"]
    }
    column_name_sets[db_id] = {
        col.lower()
        for _, col in schema["column_names_original"]
        if col != "*"
    }

# Column-table pair sets — used by alias-aware validator
def build_column_table_pairs(db_id: str) -> set:
    schema  = schema_map.get(db_id, {})
    tables  = schema.get("table_names_original", [])
    columns = schema.get("column_names_original", [])
    pairs   = set()
    for table_idx, col_name in columns:
        if table_idx == -1:
            continue
        pairs.add(f"{tables[table_idx].lower()}.{col_name.lower()}")
    return pairs

# Pre-build for ALL databases once — never rebuild inside a loop
column_table_pairs = {
    db_id: build_column_table_pairs(db_id)
    for db_id in schema_map
}

print(f"Schemas loaded:       {len(schema_map)}")
print(f"Table sets built:     {len(table_name_sets)}")
print(f"Column-pair sets:     {len(column_table_pairs)}")
# Quick sanity
sample_db = "concert_singer"
print(f"Tables in {sample_db}: {table_name_sets.get(sample_db)}")

Schemas loaded:       166
Table sets built:     166
Column-pair sets:     166
Tables in concert_singer: {'stadium', 'concert', 'singer', 'singer_in_concert'}


## 7. Load Milestone 1 and 2 Results for Cross-Milestone Comparison

Load saved CSVs so we can build the final cross-milestone table
without rerunning any previous notebook.

In [ ]:
m1_csv_path = MILESTONE1_DIR / "csv_outputs" / "milestone1_full_eval.csv"
m2_csv_path = MILESTONE2_DIR / "csv_outputs" / "milestone2_full_eval.csv"

m1_df = pd.read_csv(m1_csv_path) if m1_csv_path.exists() else None
m2_df = pd.read_csv(m2_csv_path) if m2_csv_path.exists() else None

if m1_df is not None:
    print(f"M1 results loaded: {len(m1_df)} rows")
    # Handle both possible column name variants from earlier sessions
    tuned_exec_col  = "tuned_exec_match" if "tuned_exec_match" in m1_df.columns else "tuned_exec"
    tuned_exact_col = "tuned_exact_match" if "tuned_exact_match" in m1_df.columns else "tuned_exact"
    final_exec_col  = "final_exec_match"  if "final_exec_match"  in m1_df.columns else None
    print(f"  Tuned exec acc:   {m1_df[tuned_exec_col].mean():.3f}")
    print(f"  Tuned exact acc:  {m1_df[tuned_exact_col].mean():.3f}")
    if final_exec_col:
        print(f"  Fallback exec acc:{m1_df[final_exec_col].mean():.3f}")
else:
    print("M1 CSV not found — cross-milestone table will be partial.")

if m2_df is not None:
    print(f"M2 results loaded: {len(m2_df)} rows")
    print(f"  Tuned exec acc:   {m2_df['tuned_exec'].mean():.3f}")
    print(f"  Rerank exec acc:  {m2_df['rerank_exec'].mean():.3f}")
    print(f"  Rerank exact acc: {m2_df['rerank_exact'].mean():.3f}")
else:
    print("M2 CSV not found — cross-milestone table will be partial.")

M1 results loaded: 200 rows
  Tuned exec acc:   0.525
  Tuned exact acc:  0.375
  Fallback exec acc:0.595
M2 results loaded: 200 rows
  Tuned exec acc:   0.495
  Rerank exec acc:  0.520
  Rerank exact acc: 0.380


## 8. Define All Helper Functions

All functions are defined in one cell so this notebook is fully
self-contained. Improvements over Milestone 2:
- generate_sql: repetition_penalty + no_repeat_ngram_size (stops loops)
- schema_validity_check: alias-aware dot-notation validation added
- resolve_aliases: new — parses T1/T2 → real table name mapping
- alias_aware_validity_check: new — validates alias.column pairs
- rerank_candidates: uses alias_aware_validity_check instead of basic check
- hybrid_pick: new — self-consistency among valid+executable, falls back to rerank
- generate_candidates: 8 candidates with broader temperature range

In [ ]:
from collections import Counter

SQL_KEYWORDS = {
    "select", "from", "where", "join", "on", "group", "by", "order",
    "having", "limit", "offset", "distinct", "count", "sum", "avg",
    "min", "max", "as", "and", "or", "not", "in", "like", "between",
    "is", "null", "case", "when", "then", "else", "end", "inner",
    "outer", "left", "right", "cross", "union", "all", "exists",
    "insert", "update", "delete", "create", "drop", "asc", "desc",
    "true", "false", "upper", "lower", "length", "substr", "trim",
    "coalesce", "ifnull", "strftime", "date", "time", "cast", "over",
    "partition", "row_number", "rank", "intersect", "except", "with",
}

def generate_sql(prompt, model, gen_tokenizer,
                 max_new_tokens=256, do_sample=False, temperature=1.0):
    messages = [
        {"role": "system",
         "content": "You are a text-to-SQL assistant. Return only valid SQLite SQL."},
        {"role": "user", "content": prompt},
    ]
    inputs = gen_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    )
    inputs     = {k: v.to(model.device) for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else 1.0,
            pad_token_id=gen_tokenizer.eos_token_id,
            eos_token_id=gen_tokenizer.eos_token_id,
            use_cache=True,
            repetition_penalty=1.3,
            no_repeat_ngram_size=8,
        )
    generated_ids = outputs[0, prompt_len:]
    return gen_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def clean_sql(text: str) -> str:
    text = text.strip()
    if text.lower().startswith("sql"):
        text = text[3:].strip(": \n")
    if "```" in text:
        parts = text.split("```")
        text  = parts[1] if len(parts) >= 2 else parts[0]
        text  = re.sub(r"^sql\s*", "", text.strip(), flags=re.IGNORECASE)
    text = text.split("\n\n")[0].strip()
    text = text.split(";--")[0].strip()
    return text


def normalize_sql(sql_text: str) -> str:
    sql = str(sql_text).strip().lower().replace(";", "")
    sql = re.sub(r"\s+",     " ",   sql)
    sql = re.sub(r"\s*,\s*", ", ", sql)
    sql = re.sub(r"\s*\(\s*", "(",  sql)
    sql = re.sub(r"\s*\)\s*", ")",  sql)
    sql = re.sub(r"\s*=\s*",  " = ", sql)
    return re.sub(r"\s+", " ", sql).strip()


def execute_sql(db_path: str, sql: str):
    try:
        conn = sqlite3.connect(db_path)
        cur  = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        conn.close()
        return True, rows, None
    except Exception as e:
        return False, None, str(e)


def extract_identifiers(sql_text: str) -> set:
    sql_clean = re.sub(r"'[^']*'", "", sql_text)
    sql_clean = re.sub(r'"[^"]*"', "", sql_clean)
    tokens    = re.findall(r'\b([a-zA-Z_][a-zA-Z0-9_]*)\b', sql_clean)
    return {t.lower() for t in tokens if t.lower() not in SQL_KEYWORDS}


def schema_validity_check(sql_text: str, db_id: str):
    if not sql_text or not sql_text.strip():
        return False, ["empty SQL"], 0.0
    identifiers = extract_identifiers(sql_text)
    if not identifiers:
        return True, [], 1.0
    valid_names = (table_name_sets.get(db_id, set()) |
                   column_name_sets.get(db_id, set()))
    violations  = [i for i in identifiers if i not in valid_names]
    score       = 1.0 - (len(violations) / len(identifiers))
    return len(violations) == 0, violations, round(score, 4)


def resolve_aliases(sql_text: str) -> dict:
    alias_map = {}
    # FIX: anchor to FROM/JOIN so "ON T1" can never be parsed as table=ON, alias=T1
    patterns = re.findall(
        r'(?:FROM|JOIN)\s+([a-zA-Z_][a-zA-Z0-9_]*)\s+(?:[Aa][Ss]\s+)?([tT]\d+)\b',
        sql_text,
        re.IGNORECASE
    )
    for table_name, alias in patterns:
        alias_map[alias.lower()] = table_name.lower()
    return alias_map


def alias_aware_validity_check(sql_text: str, db_id: str):
    if not sql_text or not sql_text.strip():
        return False, ["empty SQL"], 0.0

    alias_map   = resolve_aliases(sql_text)
    valid_pairs = column_table_pairs.get(db_id, set())
    valid_names = (table_name_sets.get(db_id, set()) |
                   column_name_sets.get(db_id, set()))

    # Get all identifiers but EXCLUDE alias names (t1, t2, etc.)
    # since they are not real table/column names
    alias_names = set(alias_map.keys())  # {'t1', 't2', ...}
    identifiers = extract_identifiers(sql_text) - alias_names
    basic_violations = [i for i in identifiers if i not in valid_names]

    # Check aliased dot-refs: T1.col_name
    dot_refs = re.findall(r'\b([tT]\d+)\.([a-zA-Z_][a-zA-Z0-9_]*)\b', sql_text)
    alias_violations = []
    for alias, col in dot_refs:
        actual_table = alias_map.get(alias.lower())
        if actual_table is None:
            continue
        if f"{actual_table}.{col.lower()}" not in valid_pairs:
            alias_violations.append(f"{alias}({actual_table}).{col}")

    # Check explicit dot-refs: table_name.col_name (non-alias)
    explicit_dot_refs = re.findall(
        r'\b([a-zA-Z_][a-zA-Z0-9_]*)\.([a-zA-Z_][a-zA-Z0-9_]*)\b', sql_text
    )
    explicit_violations = []
    for table_part, col_part in explicit_dot_refs:
        if re.match(r'^[tT]\d+$', table_part):
            continue  # handled above
        if f"{table_part.lower()}.{col_part.lower()}" not in valid_pairs:
            explicit_violations.append(f"{table_part}.{col_part}")

    all_violations = basic_violations + alias_violations + explicit_violations

    n_identifiers = len(identifiers)
    n_dot_checks  = len(dot_refs) + len(explicit_dot_refs)
    total_checks  = max(n_identifiers + n_dot_checks, 1)
    final_score   = round(1.0 - len(all_violations) / total_checks, 4)

    return len(all_violations) == 0, all_violations, final_score


def build_focused_schema(question: str, schema_text: str, db_id: str) -> str:
    q       = question.lower()
    tables  = schema_map[db_id]["table_names_original"]
    columns = schema_map[db_id]["column_names_original"]

    # FIX 1: use substring match (not word-split) so plurals like "singers" hit "singer"
    relevant = set()
    for idx, tbl in enumerate(tables):
        if tbl.lower() in q:
            relevant.add(idx)
    for tbl_idx, col_name in columns:
        if tbl_idx == -1:
            continue
        if col_name.lower() in q:
            relevant.add(tbl_idx)

    # FIX 2: expand via foreign keys so JOIN-partner tables are never dropped
    if relevant:
        fks      = schema_map[db_id].get("foreign_keys", [])
        col_orig = schema_map[db_id]["column_names_original"]
        changed  = True
        while changed:   # iterate until no new tables are added
            changed = False
            for src_col_idx, tgt_col_idx in fks:
                src_t = col_orig[src_col_idx][0]
                tgt_t = col_orig[tgt_col_idx][0]
                if src_t in relevant and tgt_t not in relevant:
                    relevant.add(tgt_t)
                    changed = True
                elif tgt_t in relevant and src_t not in relevant:
                    relevant.add(src_t)
                    changed = True

    if not relevant:
        return schema_text  # no signal at all — return full schema

    # Build table -> columns mapping
    table_to_cols = {i: [] for i in range(len(tables))}
    for tbl_idx, col_name in columns:
        if tbl_idx == -1:
            continue
        table_to_cols[tbl_idx].append(col_name)

    lines = ["Tables:"]
    for idx in sorted(relevant):
        lines.append(f"- {tables[idx]}({', '.join(table_to_cols[idx])})")

    # Append FK relationships for relevant tables
    fks      = schema_map[db_id].get("foreign_keys", [])
    col_orig = schema_map[db_id]["column_names_original"]
    tbl_orig = schema_map[db_id]["table_names_original"]
    if fks:
        fk_lines = []
        for src_col_idx, tgt_col_idx in fks:
            src_t, src_c = col_orig[src_col_idx]
            tgt_t, tgt_c = col_orig[tgt_col_idx]
            if src_t in relevant or tgt_t in relevant:
                fk_lines.append(f"- {tbl_orig[src_t]}.{src_c} -> {tbl_orig[tgt_t]}.{tgt_c}")
        if fk_lines:
            lines.append("\nForeign Keys:")
            lines.extend(fk_lines)

    return "\n".join(lines)


def generate_candidates(prompt: str, model, gen_tokenizer, n_candidates: int = 8) -> list:
    candidates = [clean_sql(generate_sql(prompt, model, gen_tokenizer, do_sample=False))]
    for temp in [0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0][:n_candidates - 1]:
        candidates.append(clean_sql(generate_sql(
            prompt, model, gen_tokenizer, do_sample=True, temperature=temp
        )))
    return candidates


def rerank_candidates(candidates: list, db_id: str, db_path: str):
    scored = []
    for rank, sql in enumerate(candidates):
        is_valid, violations, schema_score = alias_aware_validity_check(sql, db_id)
        exec_ok, rows, exec_err = execute_sql(db_path, sql)
        scored.append({
            "sql": sql, "schema_score": schema_score, "is_valid": is_valid,
            "exec_ok": exec_ok, "exec_err": exec_err, "violations": violations,
            "row_count": len(rows) if exec_ok and rows else 0, "original_rank": rank,
        })
    scored.sort(key=lambda x: (-x["schema_score"], -int(x["exec_ok"]), x["original_rank"]))
    return scored[0]["sql"], scored


def self_consistency_pick(candidates: list) -> str:
    if not candidates:
        return ""
    normalized  = [normalize_sql(c) for c in candidates]
    most_common = Counter(normalized).most_common(1)[0][0]
    for c in candidates:
        if normalize_sql(c) == most_common:
            return c
    return candidates[0]


def hybrid_pick(candidates: list, db_id: str, db_path: str) -> str:
    valid_cands      = [c for c in candidates if alias_aware_validity_check(c, db_id)[0]]
    exec_valid_cands = [c for c in valid_cands if execute_sql(db_path, c)[0]]
    if exec_valid_cands:
        return self_consistency_pick(exec_valid_cands)
    elif valid_cands:
        return self_consistency_pick(valid_cands)
    else:
        best, _ = rerank_candidates(candidates, db_id, db_path)
        return best


print("All helper functions defined correctly.")
print(f"  SQL keywords: {len(SQL_KEYWORDS)}")

# Sanity checks
test = normalize_sql("SELECT   count(*)   FROM  singer  WHERE age>20")
assert "  " not in test, f"normalize_sql broken: '{test}'"
print(f"  normalize_sql sanity check passed: '{test}'")

aliases = resolve_aliases("SELECT T1.name FROM singer AS T1 JOIN concert AS T2 ON T1.id = T2.id")
assert aliases == {"t1": "singer", "t2": "concert"}, f"resolve_aliases broken: {aliases}"
print(f"  resolve_aliases sanity check passed: {aliases}")

# Focused schema sanity: pluralised question should still find the table
_test_db = list(schema_map.keys())[0]
_test_q  = list(schema_map[_test_db]["table_names_original"])[0] + "s are great"
_result  = build_focused_schema(_test_q, "fallback", _test_db)
assert _result != "fallback", "build_focused_schema substring match broken"
print(f"  build_focused_schema substring sanity check passed")

All helper functions defined correctly.
  SQL keywords: 66
  normalize_sql sanity check passed: 'select count(*)from singer where age>20'
  resolve_aliases sanity check passed: {'t1': 'singer', 't2': 'concert'}
  build_focused_schema substring sanity check passed


## 9. Load Tokenizer and Both Models

Two completely separate model instances — base model (no adapter)
and LoRA-tuned model. Loading them separately guarantees no adapter
weight leaks into the base model comparison.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import gc, torch

MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded.")

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
)
base_model.eval()
print("Base model loaded.")

# Load LoRA on a fresh base, then merge weights into a flat model
lora_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
)
tuned_model = PeftModel.from_pretrained(lora_base, str(ADAPTER_DIR))
tuned_model = tuned_model.merge_and_unload()
tuned_model.eval()
print("LoRA tuned model loaded and merged.")

gc.collect()
torch.cuda.empty_cache()

print("Same object?", base_model is tuned_model)
assert base_model is not tuned_model
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(f"GPU memory reserved:  {torch.cuda.memory_reserved()/1e9:.1f} GB")

Tokenizer loaded.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Base model loaded.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LoRA tuned model loaded and merged.
Same object? False
GPU memory allocated: 15.3 GB
GPU memory reserved:  15.3 GB


## 10. Quick Sanity Check

Verify all components work on one example before running the full loop.

In [ ]:
import re # Ensure re is imported

# Rest of the original code
sample_ex = processed_dev[0]
print("DB:      ", sample_ex["db_id"])
print("Question:", sample_ex["question"])
print("Gold SQL:", sample_ex["target_sql"])

# Focused schema
focused = build_focused_schema(
    sample_ex["question"], sample_ex["schema_text"], sample_ex["db_id"]
)
print("\nFocused schema:\n", focused)

# Alias-aware validity on gold SQL
is_v, viol, score = alias_aware_validity_check(
    sample_ex["target_sql"], sample_ex["db_id"]
)
print(f"\nGold SQL validity: {is_v} | score={score} | violations={viol}")

# Generate 3 candidates (quick test — full run uses 8)
print("\nGenerating 3 test candidates...")
test_cands = generate_candidates(
    sample_ex["prompt"], tuned_model, tokenizer, n_candidates=3
)
for j, c in enumerate(test_cands):
    v, _, s = alias_aware_validity_check(c, sample_ex["db_id"])
    ok, _, _ = execute_sql(sample_ex["sqlite_path"], c)
    print(f"  Cand {j}: valid={v} score={s} exec={ok} | {c[:70]}")

hybrid = hybrid_pick(test_cands, sample_ex["db_id"], sample_ex["sqlite_path"])
print("\nHybrid pick:", hybrid)

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48


DB:       concert_singer
Question: How many singers do we have?
Gold SQL: SELECT count(*) FROM singer

Focused schema:
 Tables:
- stadium(Stadium_ID, Location, Name, Capacity, Highest, Lowest, Average)
- singer(Singer_ID, Name, Country, Song_Name, Song_release_year, Age, Is_male)
- concert(concert_ID, concert_Name, Theme, Stadium_ID, Year)
- singer_in_concert(concert_ID, Singer_ID)

Foreign Keys:
- concert.Stadium_ID -> stadium.Stadium_ID
- singer_in_concert.Singer_ID -> singer.Singer_ID
- singer_in_concert.concert_ID -> concert.concert_ID

Gold SQL validity: True | score=1.0 | violations=[]

Generating 3 test candidates...


  Cand 0: valid=True score=1.0 exec=True | SELECT count(*) FROM  "singer"
  Cand 1: valid=True score=1.0 exec=True | SELECT count(*) FROM  "singer"
  Cand 2: valid=True score=1.0 exec=True | SELECT count(*) FROM  "singer"

Hybrid pick: SELECT count(*) FROM  "singer"


## 11. Full Evaluation Loop

Evaluates 6 systems on every dev example:
  1. Base model (no fine-tuning)
  2. Tuned greedy (Milestone 1 equivalent)
  3. Schema validity filter (first alias-aware-valid candidate)
  4. Reranker (alias-aware schema score + executability)
  5. Self-consistency (most common candidate after normalisation)
  6. Hybrid picker (self-consistency among valid+executable, else rerank)

Per-example timing and checkpoint every 25 examples protect against
Colab disconnections. Resume logic included below.

In [ ]:
import os
chk = CSV_DIR / "m3_checkpoint.csv"
if chk.exists():
    os.remove(chk)
    print("Old checkpoint cleared.")

Old checkpoint cleared.


In [ ]:
N_CANDIDATES = 8
DEV_LIMIT    = 275

validation_examples = processed_dev[:DEV_LIMIT]
full_rows  = []
loop_start = time.time()

print(f"Evaluating {DEV_LIMIT} examples | {N_CANDIDATES} candidates each")
print(f"Systems: base | tuned | schema_filter | reranker | consistency | hybrid")
print(f"{'='*75}")

for i, ex in enumerate(validation_examples):
    t0 = time.time()
    print(f"Ex {i:>4}/{DEV_LIMIT} | {ex['db_id']:<22}", end=" ", flush=True)

    gold_sql = ex["target_sql"]
    db_id    = ex["db_id"]
    db_path  = ex["sqlite_path"]

    # Use the original prompt as-is — the model was fine-tuned on this exact format
    prompt = ex["prompt"]

    # ── System 1: Base model ──────────────────────────────────────────────
    base_sql = clean_sql(generate_sql(prompt, base_model, tokenizer))
    print("base✓", end=" ", flush=True)

    # ── System 2: Tuned greedy ────────────────────────────────────────────
    tuned_sql = clean_sql(generate_sql(prompt, tuned_model, tokenizer))
    print("tuned✓", end=" ", flush=True)

    # ── Generate 8 candidates (shared for systems 3-6) ────────────────────
    candidates = generate_candidates(prompt, tuned_model, tokenizer, N_CANDIDATES)
    print("cands✓", end=" ", flush=True)

    # ── System 3: Schema filter ───────────────────────────────────────────
    schema_best = tuned_sql
    for cand in candidates:
        if alias_aware_validity_check(cand, db_id)[0]:
            schema_best = cand
            break

    # ── System 4: Reranker ────────────────────────────────────────────────
    reranked_sql, scored_cands = rerank_candidates(candidates, db_id, db_path)

    # ── System 5: Self-consistency ────────────────────────────────────────
    consistency_sql = self_consistency_pick(candidates)

    # ── System 6: Hybrid picker ───────────────────────────────────────────
    hybrid_sql = hybrid_pick(candidates, db_id, db_path)
    print("all✓", end=" ", flush=True)

    # ── Execute all systems ───────────────────────────────────────────────
    gold_ok,    gold_res,    _             = execute_sql(db_path, gold_sql)
    base_ok,    base_res,    base_err      = execute_sql(db_path, base_sql)
    tuned_ok,   tuned_res,   tuned_err     = execute_sql(db_path, tuned_sql)
    schema_ok,  schema_res,  schema_err    = execute_sql(db_path, schema_best)
    rerank_ok,  rerank_res,  rerank_err    = execute_sql(db_path, reranked_sql)
    consist_ok, consist_res, consist_err   = execute_sql(db_path, consistency_sql)
    hybrid_ok,  hybrid_res,  hybrid_err    = execute_sql(db_path, hybrid_sql)

    # ── Execution match (same result set as gold) ─────────────────────────
    base_exec    = gold_ok and base_ok    and gold_res == base_res
    tuned_exec   = gold_ok and tuned_ok   and gold_res == tuned_res
    schema_exec  = gold_ok and schema_ok  and gold_res == schema_res
    rerank_exec  = gold_ok and rerank_ok  and gold_res == rerank_res
    consist_exec = gold_ok and consist_ok and gold_res == consist_res
    hybrid_exec  = gold_ok and hybrid_ok  and gold_res == hybrid_res

    # ── Exact string match (normalised) ───────────────────────────────────
    gn = normalize_sql(gold_sql)
    base_exact    = normalize_sql(base_sql)        == gn
    tuned_exact   = normalize_sql(tuned_sql)       == gn
    schema_exact  = normalize_sql(schema_best)     == gn
    rerank_exact  = normalize_sql(reranked_sql)    == gn
    consist_exact = normalize_sql(consistency_sql) == gn
    hybrid_exact  = normalize_sql(hybrid_sql)      == gn

    # ── Alias-aware schema validity scores ────────────────────────────────
    _, _, base_sv    = alias_aware_validity_check(base_sql,       db_id)
    _, _, tuned_sv   = alias_aware_validity_check(tuned_sql,      db_id)
    _, _, schema_sv  = alias_aware_validity_check(schema_best,    db_id)
    _, _, rerank_sv  = alias_aware_validity_check(reranked_sql,   db_id)
    _, _, consist_sv = alias_aware_validity_check(consistency_sql,db_id)
    _, _, hybrid_sv  = alias_aware_validity_check(hybrid_sql,     db_id)

    full_rows.append({
        "example_id":    i,
        "db_id":         db_id,
        "question":      ex["question"],
        "gold_sql":      gold_sql,
        "sqlite_path":   db_path,
        "base_sql":      base_sql,
        "tuned_sql":     tuned_sql,
        "schema_sql":    schema_best,
        "rerank_sql":    reranked_sql,
        "consist_sql":   consistency_sql,
        "hybrid_sql":    hybrid_sql,
        "base_exec":     base_exec,
        "tuned_exec":    tuned_exec,
        "schema_exec":   schema_exec,
        "rerank_exec":   rerank_exec,
        "consist_exec":  consist_exec,
        "hybrid_exec":   hybrid_exec,
        "base_exact":    base_exact,
        "tuned_exact":   tuned_exact,
        "schema_exact":  schema_exact,
        "rerank_exact":  rerank_exact,
        "consist_exact": consist_exact,
        "hybrid_exact":  hybrid_exact,
        "base_sv":       base_sv,
        "tuned_sv":      tuned_sv,
        "schema_sv":     schema_sv,
        "rerank_sv":     rerank_sv,
        "consist_sv":    consist_sv,
        "hybrid_sv":     hybrid_sv,
        "base_err":      base_err,
        "tuned_err":     tuned_err,
        "schema_err":    schema_err,
        "rerank_err":    rerank_err,
        "consist_err":   consist_err,
        "hybrid_err":    hybrid_err,
    })

    t1 = time.time()
    print(f"| {t1-t0:.1f}s")

    if (i + 1) % 25 == 0:
        elapsed = time.time() - loop_start
        avg     = elapsed / (i + 1)
        eta     = avg * (DEV_LIMIT - (i + 1))
        chk     = pd.DataFrame(full_rows)
        chk.to_csv(CSV_DIR / "m3_checkpoint.csv", index=False)
        torch.cuda.empty_cache()
        print(f"\n{'─'*75}")
        print(f"  {i+1}/{DEV_LIMIT} | "
              f"Elapsed: {elapsed/60:.1f}min | "
              f"ETA: {eta/60:.1f}min | checkpoint saved")
        print(f"  Running hybrid exec acc: "
              f"{sum(r['hybrid_exec'] for r in full_rows)/len(full_rows):.3f}")
        print(f"  Running tuned exact acc: "
              f"{sum(r['tuned_exact'] for r in full_rows)/len(full_rows):.3f}")
        print(f"{'─'*75}\n")

eval_df = pd.DataFrame(full_rows)
total   = len(eval_df)
elapsed = time.time() - loop_start
print(f"\nFinished {total} examples in {elapsed/60:.1f} minutes.")

Evaluating 275 examples | 8 candidates each
Systems: base | tuned | schema_filter | reranker | consistency | hybrid
Ex    0/275 | concert_singer         base✓ tuned✓ cands✓ all✓ | 20.2s
Ex    1/275 | concert_singer         base✓ tuned✓ cands✓ all✓ | 34.8s
Ex    2/275 | concert_singer         base✓ tuned✓ cands✓ all✓ | 25.1s
Ex    3/275 | concert_singer         base✓ tuned✓ cands✓ all✓ | 23.0s
Ex    4/275 | concert_singer         base✓ tuned✓ cands✓ all✓ | 27.9s
Ex    5/275 | concert_singer         base✓ tuned✓ cands✓ all✓ | 41.8s
Ex    6/275 | concert_singer         base✓ tuned✓ cands✓ all✓ | 123.7s
Ex    7/275 | concert_singer         base✓ tuned✓ cands✓ all✓ | 26.1s
Ex    8/275 | concert_singer         base✓ tuned✓ cands✓ all✓ | 108.0s
Ex    9/275 | concert_singer         base✓ tuned✓ cands✓ all✓ | 54.4s
Ex   10/275 | concert_singer         base✓ tuned✓ cands✓ all✓ | 94.0s
Ex   11/275 | concert_singer         base✓ tuned✓ cands✓ all✓ | 107.2s
Ex   12/275 | concert_singer         base

## 11A. Resume After Disconnect (Only Run If Loop Was Interrupted)

If Colab disconnected mid-loop, run this cell to reload the checkpoint
and continue from where you left off. Skip this cell otherwise.

In [ ]:
# ONLY RUN THIS IF THE LOOP WAS INTERRUPTED

checkpoint_path = CSV_DIR / "m3_checkpoint.csv"
assert checkpoint_path.exists(), "No checkpoint found — start from the beginning."

checkpoint_df   = pd.read_csv(checkpoint_path)
completed_ids   = set(checkpoint_df["example_id"].tolist())
full_rows       = checkpoint_df.to_dict("records")

print(f"Resuming from example {max(completed_ids) + 1}")
print(f"Already completed: {len(completed_ids)} / {len(processed_dev)}")

N_CANDIDATES = 8
DEV_LIMIT    = 275
loop_start   = time.time()

for i, ex in enumerate(processed_dev[:DEV_LIMIT]):
    if i in completed_ids:
        continue

    t0      = time.time()
    db_id   = ex["db_id"]
    db_path = ex["sqlite_path"]
    print(f"Ex {i:>4}/{DEV_LIMIT} | {db_id:<22}", end=" ", flush=True)

    # Use the original prompt as-is
    prompt   = ex["prompt"]
    gold_sql = ex["target_sql"]

    base_sql   = clean_sql(generate_sql(prompt, base_model, tokenizer))
    print("base✓", end=" ", flush=True)
    tuned_sql  = clean_sql(generate_sql(prompt, tuned_model, tokenizer))
    print("tuned✓", end=" ", flush=True)
    candidates = generate_candidates(prompt, tuned_model, tokenizer, N_CANDIDATES)
    print("cands✓", end=" ", flush=True)

    schema_best     = next((c for c in candidates if alias_aware_validity_check(c, db_id)[0]), tuned_sql)
    reranked_sql, _ = rerank_candidates(candidates, db_id, db_path)
    consistency_sql = self_consistency_pick(candidates)
    hybrid_sql      = hybrid_pick(candidates, db_id, db_path)
    print("all✓", end=" ", flush=True)

    gold_ok,    gold_res,    _            = execute_sql(db_path, gold_sql)
    base_ok,    base_res,    base_err     = execute_sql(db_path, base_sql)
    tuned_ok,   tuned_res,   tuned_err    = execute_sql(db_path, tuned_sql)
    schema_ok,  schema_res,  schema_err   = execute_sql(db_path, schema_best)
    rerank_ok,  rerank_res,  rerank_err   = execute_sql(db_path, reranked_sql)
    consist_ok, consist_res, consist_err  = execute_sql(db_path, consistency_sql)
    hybrid_ok,  hybrid_res,  hybrid_err   = execute_sql(db_path, hybrid_sql)

    gn = normalize_sql(gold_sql)
    _, _, base_sv    = alias_aware_validity_check(base_sql,        db_id)
    _, _, tuned_sv   = alias_aware_validity_check(tuned_sql,       db_id)
    _, _, schema_sv  = alias_aware_validity_check(schema_best,     db_id)
    _, _, rerank_sv  = alias_aware_validity_check(reranked_sql,    db_id)
    _, _, consist_sv = alias_aware_validity_check(consistency_sql, db_id)
    _, _, hybrid_sv  = alias_aware_validity_check(hybrid_sql,      db_id)

    full_rows.append({
        "example_id": i, "db_id": db_id, "question": ex["question"],
        "gold_sql": gold_sql, "sqlite_path": db_path,
        "base_sql": base_sql, "tuned_sql": tuned_sql,
        "schema_sql": schema_best, "rerank_sql": reranked_sql,
        "consist_sql": consistency_sql, "hybrid_sql": hybrid_sql,
        "base_exec":    gold_ok and base_ok    and gold_res == base_res,
        "tuned_exec":   gold_ok and tuned_ok   and gold_res == tuned_res,
        "schema_exec":  gold_ok and schema_ok  and gold_res == schema_res,
        "rerank_exec":  gold_ok and rerank_ok  and gold_res == rerank_res,
        "consist_exec": gold_ok and consist_ok and gold_res == consist_res,
        "hybrid_exec":  gold_ok and hybrid_ok  and gold_res == hybrid_res,
        "base_exact":    normalize_sql(base_sql)        == gn,
        "tuned_exact":   normalize_sql(tuned_sql)       == gn,
        "schema_exact":  normalize_sql(schema_best)     == gn,
        "rerank_exact":  normalize_sql(reranked_sql)    == gn,
        "consist_exact": normalize_sql(consistency_sql) == gn,
        "hybrid_exact":  normalize_sql(hybrid_sql)      == gn,
        "base_sv": base_sv, "tuned_sv": tuned_sv, "schema_sv": schema_sv,
        "rerank_sv": rerank_sv, "consist_sv": consist_sv, "hybrid_sv": hybrid_sv,
        "base_err": base_err, "tuned_err": tuned_err, "schema_err": schema_err,
        "rerank_err": rerank_err, "consist_err": consist_err, "hybrid_err": hybrid_err,
    })

    print(f"| {time.time()-t0:.1f}s")

    if (i + 1) % 25 == 0:
        pd.DataFrame(full_rows).to_csv(CSV_DIR / "m3_checkpoint.csv", index=False)
        print(f"  Checkpoint saved at {i+1}")

eval_df = pd.DataFrame(full_rows).sort_values("example_id").reset_index(drop=True)
total   = len(eval_df)
print(f"\nResume complete. Total examples: {total}")

## 12. Final Ablation Table

The headline result for the report. Shows each component's
incremental contribution across execution accuracy, exact match,
and mean alias-aware schema validity score.

In [ ]:
total = len(eval_df)

print("=" * 72)
print(f"{'System':<35} {'Exec Acc':>10} {'Exact':>8} {'Mean SV':>9}")
print("=" * 72)

systems = [
    ("1. Base model (no tuning)",          "base_exec",    "base_exact",    "base_sv"),
    ("2. + LoRA fine-tuning (M1)",         "tuned_exec",   "tuned_exact",   "tuned_sv"),
    ("3. + Schema filter (M2)",            "schema_exec",  "schema_exact",  "schema_sv"),
    ("4. + Reranker (M2)",                 "rerank_exec",  "rerank_exact",  "rerank_sv"),
    ("5. + Self-consistency (M2)",         "consist_exec", "consist_exact", "consist_sv"),
    ("6. + Hybrid picker (M3) ← BEST",    "hybrid_exec",  "hybrid_exact",  "hybrid_sv"),
]

for name, exec_col, exact_col, sv_col in systems:
    ea  = eval_df[exec_col].mean()
    em  = eval_df[exact_col].mean()
    sv  = eval_df[sv_col].mean()
    print(f"{name:<35} {ea:>9.1%} {em:>8.1%} {sv:>9.4f}")

print("=" * 72)
print(f"Total examples: {total}")

print("\nSchema validity — fully valid SQL count (alias-aware):")
for name, sv_col in [
    ("Base",         "base_sv"),
    ("Tuned",        "tuned_sv"),
    ("Schema filter","schema_sv"),
    ("Reranked",     "rerank_sv"),
    ("Hybrid",       "hybrid_sv"),
]:
    count = (eval_df[sv_col] == 1.0).sum()
    print(f"  {name:<15}: {count:>4} / {total}  ({count/total:.1%})")

System                                Exec Acc    Exact   Mean SV
1. Base model (no tuning)               15.6%     0.7%    0.2335
2. + LoRA fine-tuning (M1)              13.8%     8.7%    0.4364
3. + Schema filter (M2)                 19.3%    12.4%    0.5159
4. + Reranker (M2)                      20.7%    13.1%    0.6421
5. + Self-consistency (M2)              14.2%     9.1%    0.4434
6. + Hybrid picker (M3) ← BEST          20.7%    13.1%    0.6421
Total examples: 275

Schema validity — fully valid SQL count (alias-aware):
  Base           :    9 / 275  (3.3%)
  Tuned          :   48 / 275  (17.5%)
  Schema filter  :   86 / 275  (31.3%)
  Reranked       :   86 / 275  (31.3%)
  Hybrid         :   86 / 275  (31.3%)


## 13. Cross-Milestone Comparison

Compare the 200-example subset numbers from M1 and M2 against
M3's full evaluation to confirm consistency and show progress.

In [ ]:
m3_200 = eval_df[:200]  # first 200 for apples-to-apples

print("Cross-Milestone Comparison")
print("=" * 68)
print(f"{'Metric':<38} {'M1(200)':>9} {'M2(200)':>9} {'M3(200)':>9} {'M3(all)':>9}")
print("=" * 68)

def get(df, col, fallback=None):
    if df is None or col not in df.columns:
        return fallback
    return df[col].mean()

rows = [
    ("Tuned exec acc",
        get(m1_df, tuned_exec_col,  "N/A"),
        get(m2_df, "tuned_exec",    "N/A"),
        f"{m3_200['tuned_exec'].mean():.3f}",
        f"{eval_df['tuned_exec'].mean():.3f}"),
    ("Best system exec acc",
        get(m1_df, final_exec_col,  "N/A") if final_exec_col else "N/A",
        get(m2_df, "rerank_exec",   "N/A"),
        f"{m3_200['hybrid_exec'].mean():.3f}",
        f"{eval_df['hybrid_exec'].mean():.3f}"),
    ("Tuned exact acc",
        get(m1_df, tuned_exact_col, "N/A"),
        get(m2_df, "tuned_exact",   "N/A"),
        f"{m3_200['tuned_exact'].mean():.3f}",
        f"{eval_df['tuned_exact'].mean():.3f}"),
    ("Best system exact acc",
        "N/A",
        get(m2_df, "rerank_exact",  "N/A"),
        f"{m3_200['hybrid_exact'].mean():.3f}",
        f"{eval_df['hybrid_exact'].mean():.3f}"),
    ("Mean schema validity (best)",
        "N/A",
        get(m2_df, "mean_rerank_schema_score", "N/A")
            if m2_df is not None and "rerank_schema_score" in m2_df.columns
            else "N/A",
        f"{m3_200['hybrid_sv'].mean():.4f}",
        f"{eval_df['hybrid_sv'].mean():.4f}"),
]

for row in rows:
    name = row[0]
    vals = [f"{v:.3f}" if isinstance(v, float) else str(v) for v in row[1:]]
    print(f"{name:<38} {vals[0]:>9} {vals[1]:>9} {vals[2]:>9} {vals[3]:>9}")

print("=" * 68)

Cross-Milestone Comparison
Metric                                   M1(200)   M2(200)   M3(200)   M3(all)
Tuned exec acc                             0.525     0.495     0.145     0.138
Best system exec acc                       0.595     0.520     0.220     0.207
Tuned exact acc                            0.375     0.375     0.090     0.087
Best system exact acc                        N/A     0.380     0.140     0.131
Mean schema validity (best)                  N/A       N/A    0.6879    0.6421


## 14. Head-to-Head Analysis

Detailed breakdown of where the hybrid system wins and loses
compared to the Milestone 1 tuned-greedy baseline.

In [ ]:
# Hybrid vs tuned greedy
hybrid_better = eval_df[ eval_df["hybrid_exec"] & ~eval_df["tuned_exec"]]
tuned_better  = eval_df[~eval_df["hybrid_exec"] &  eval_df["tuned_exec"]]
both_correct  = eval_df[ eval_df["hybrid_exec"] &  eval_df["tuned_exec"]]
both_wrong    = eval_df[~eval_df["hybrid_exec"] & ~eval_df["tuned_exec"]]

print("Hybrid Picker vs Tuned-Greedy (exec):")
print(f"  Hybrid better:  {len(hybrid_better):>4}  ({len(hybrid_better)/total:.1%})")
print(f"  Tuned better:   {len(tuned_better):>4}  ({len(tuned_better)/total:.1%})")
print(f"  Both correct:   {len(both_correct):>4}  ({len(both_correct)/total:.1%})")
print(f"  Both wrong:     {len(both_wrong):>4}  ({len(both_wrong)/total:.1%})")
print(f"  Net gain:       {len(hybrid_better) - len(tuned_better):>+4}")

# Reranker vs tuned greedy
rerank_better = eval_df[ eval_df["rerank_exec"] & ~eval_df["tuned_exec"]]
rerank_worse  = eval_df[~eval_df["rerank_exec"] &  eval_df["tuned_exec"]]
print(f"\nReranker vs Tuned-Greedy net gain: {len(rerank_better)-len(rerank_worse):>+4}")

# Schema validity progression
print("\nAlias-aware fully valid SQL (score = 1.0):")
for sys_name, col in [
    ("Base",          "base_sv"),
    ("Tuned",         "tuned_sv"),
    ("Schema filter", "schema_sv"),
    ("Reranked",      "rerank_sv"),
    ("Hybrid",        "hybrid_sv"),
]:
    n = (eval_df[col] == 1.0).sum()
    print(f"  {sys_name:<16}: {n:>4} / {total}  ({n/total:.1%})")

Hybrid Picker vs Tuned-Greedy (exec):
  Hybrid better:    20  (7.3%)
  Tuned better:      1  (0.4%)
  Both correct:     37  (13.5%)
  Both wrong:      217  (78.9%)
  Net gain:        +19

Reranker vs Tuned-Greedy net gain:  +19

Alias-aware fully valid SQL (score = 1.0):
  Base            :    9 / 275  (3.3%)
  Tuned           :   48 / 275  (17.5%)
  Schema filter   :   86 / 275  (31.3%)
  Reranked        :   86 / 275  (31.3%)
  Hybrid          :   86 / 275  (31.3%)


## 15. Error Analysis

Categorise all remaining failures of the best system (hybrid),
analyse by SQL complexity, and identify the top databases
causing failures.

In [ ]:
failures  = eval_df[~eval_df["hybrid_exec"]].copy()
successes = eval_df[ eval_df["hybrid_exec"]].copy()

print(f"Hybrid failures:  {len(failures)} / {total}  ({len(failures)/total:.1%})")
print(f"Hybrid successes: {len(successes)} / {total}  ({len(successes)/total:.1%})")

# Error category classification
def classify_error(err_msg):
    if err_msg is None:
        return "wrong_result_set"
    e = str(err_msg).lower()
    if "no such column"   in e: return "wrong_column"
    if "no such table"    in e: return "wrong_table"
    if "syntax error"     in e: return "syntax_error"
    if "incomplete input" in e: return "incomplete_sql"
    if "ambiguous"        in e: return "ambiguous_column"
    if "no such function" in e: return "wrong_function"
    return "other_error"

failures["error_category"] = failures["hybrid_err"].apply(classify_error)

print("\nError categories in hybrid failures:")
print(failures["error_category"].value_counts().to_string())

print("\nTop 10 databases with most failures:")
print(failures["db_id"].value_counts().head(10).to_string())

# SQL structural complexity: JOIN count
def count_joins(sql):
    return str(sql).lower().count(" join ")

failures["gold_joins"]  = failures["gold_sql"].apply(count_joins)
successes["gold_joins"] = successes["gold_sql"].apply(count_joins)

print(f"\nAvg JOINs in gold SQL:")
print(f"  Failures:  {failures['gold_joins'].mean():.2f}")
print(f"  Successes: {successes['gold_joins'].mean():.2f}")

# Nested subquery count
def count_nested(sql):
    return max(0, str(sql).lower().count("select") - 1)

failures["nested"]  = failures["gold_sql"].apply(count_nested)
successes["nested"] = successes["gold_sql"].apply(count_nested)

print(f"\nAvg nested SELECT in gold SQL:")
print(f"  Failures:  {failures['nested'].mean():.2f}")
print(f"  Successes: {successes['nested'].mean():.2f}")

print("\nTop 15 specific errors in hybrid failures:")
top_errors = failures["hybrid_err"].dropna().value_counts().head(15)
print(top_errors.to_string())

Hybrid failures:  218 / 275  (79.3%)
Hybrid successes: 57 / 275  (20.7%)

Error categories in hybrid failures:
error_category
syntax_error        79
wrong_table         64
wrong_column        39
wrong_result_set    18
other_error         15
ambiguous_column     3

Top 10 databases with most failures:
db_id
car_1                       84
flight_2                    64
pets_1                      34
concert_singer              29
employee_hire_evaluation     7

Avg JOINs in gold SQL:
  Failures:  0.92
  Successes: 0.05

Avg nested SELECT in gold SQL:
  Failures:  0.18
  Successes: 0.05

Top 15 specific errors in hybrid failures:
hybrid_err
no such table: models       9
near "AS": syntax error     9
no such table: CarNames     7
no such table: cars         6
near ")": syntax error      5
unrecognized token: "\"     5
near "*": syntax error      4
near "=": syntax error      4
near "+": syntax error      4
no such column: Name        4
no such table: students     3
near "(": syntax error  

## 16. Qualitative Examples

Show side-by-side examples where the hybrid system succeeded
but the base model failed, and vice versa. Useful for the report.

In [ ]:
# Cases where hybrid is better
hybrid_wins = eval_df[
    eval_df["hybrid_exec"] & ~eval_df["tuned_exec"]
].head(5)

print("=" * 80)
print("EXAMPLES WHERE HYBRID PICKER BEATS TUNED-GREEDY")
print("=" * 80)
for _, row in hybrid_wins.iterrows():
    print(f"\nDB: {row['db_id']}")
    print(f"Q:  {row['question']}")
    print(f"Gold:   {row['gold_sql']}")
    print(f"Tuned:  {row['tuned_sql']}")
    print(f"Hybrid: {row['hybrid_sql']}")
    print("-" * 60)

# Cases where tuned greedy is better (regressions)
tuned_wins = eval_df[
    eval_df["tuned_exec"] & ~eval_df["hybrid_exec"]
].head(5)

print("\n" + "=" * 80)
print("REGRESSIONS — TUNED-GREEDY BEATS HYBRID PICKER")
print("=" * 80)
for _, row in tuned_wins.iterrows():
    print(f"\nDB: {row['db_id']}")
    print(f"Q:  {row['question']}")
    print(f"Gold:   {row['gold_sql']}")
    print(f"Tuned:  {row['tuned_sql']}")
    print(f"Hybrid: {row['hybrid_sql']}")
    print(f"Err:    {row['hybrid_err']}")
    print("-" * 60)

EXAMPLES WHERE HYBRID PICKER BEATS TUNED-GREEDY

DB: concert_singer
Q:  What is the name and capacity for the stadium with highest average attendance?
Gold:   SELECT name ,  capacity FROM stadium ORDER BY average DESC LIMIT 1
Tuned:  SELECT NAME ,  CAPACITY FROM stadium ORDER BY avgage DESC LIMIT 1
Hybrid: SELECT NAME ,  CAPACITY FROM stadium ORDER BY average DESC LIMIT 1
------------------------------------------------------------

DB: concert_singer
Q:  What is the name and capacity for the stadium with the highest average attendance?
Gold:   SELECT name ,  capacity FROM stadium ORDER BY average DESC LIMIT 1
Tuned:  SELECT NAME ,  CAPACITY FROM stadium ORDER BY averge DESC LIMIT 1
Hybrid: SELECT NAME ,  CAPACITY FROM stadium ORDER BY average DESC LIMIT 1
------------------------------------------------------------

DB: concert_singer
Q:  What are the names of the singers who performed in a concert in 2014?
Gold:   SELECT T2.name FROM singer_in_concert AS T1 JOIN singer AS T2 ON T1.si

## 17. Save All Milestone 3 Results to Drive

In [ ]:
# Full evaluation CSV
full_csv = CSV_DIR / "milestone3_full_eval.csv"
eval_df.to_csv(full_csv, index=False)
print("Saved full eval:", full_csv)

# Compact summary JSON — primary input for Milestone 4 report notebook
summary = {
    "total_examples":      total,
    "eval_time_minutes": round(elapsed / 60, 1) if 'elapsed' in dir() else -1,
    # Execution accuracy
    "base_exec_acc":       round(eval_df["base_exec"].mean(),    4),
    "tuned_exec_acc":      round(eval_df["tuned_exec"].mean(),   4),
    "schema_exec_acc":     round(eval_df["schema_exec"].mean(),  4),
    "rerank_exec_acc":     round(eval_df["rerank_exec"].mean(),  4),
    "consist_exec_acc":    round(eval_df["consist_exec"].mean(), 4),
    "hybrid_exec_acc":     round(eval_df["hybrid_exec"].mean(),  4),
    # Exact match accuracy
    "base_exact_acc":      round(eval_df["base_exact"].mean(),    4),
    "tuned_exact_acc":     round(eval_df["tuned_exact"].mean(),   4),
    "schema_exact_acc":    round(eval_df["schema_exact"].mean(),  4),
    "rerank_exact_acc":    round(eval_df["rerank_exact"].mean(),  4),
    "consist_exact_acc":   round(eval_df["consist_exact"].mean(), 4),
    "hybrid_exact_acc":    round(eval_df["hybrid_exact"].mean(),  4),
    # Schema validity (alias-aware)
    "mean_base_sv":        round(eval_df["base_sv"].mean(),    4),
    "mean_tuned_sv":       round(eval_df["tuned_sv"].mean(),   4),
    "mean_rerank_sv":      round(eval_df["rerank_sv"].mean(),  4),
    "mean_hybrid_sv":      round(eval_df["hybrid_sv"].mean(),  4),
    # Fully valid SQL counts
    "fully_valid_base":    int((eval_df["base_sv"]   == 1.0).sum()),
    "fully_valid_tuned":   int((eval_df["tuned_sv"]  == 1.0).sum()),
    "fully_valid_hybrid":  int((eval_df["hybrid_sv"] == 1.0).sum()),
    # Head-to-head
    "hybrid_beats_tuned":  int(( eval_df["hybrid_exec"] & ~eval_df["tuned_exec"]).sum()),
    "tuned_beats_hybrid":  int((~eval_df["hybrid_exec"] &  eval_df["tuned_exec"]).sum()),
    "both_correct":        int(( eval_df["hybrid_exec"] &  eval_df["tuned_exec"]).sum()),
    "both_wrong":          int((~eval_df["hybrid_exec"] & ~eval_df["tuned_exec"]).sum()),
}

summary_path = CSV_DIR / "milestone3_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print("Saved summary:  ", summary_path)

print("\nMilestone 3 Final Summary:")
print(f"  {'System':<25} {'Exec Acc':>10} {'Exact Acc':>10} {'Mean SV':>10}")
print(f"  {'-'*57}")
for sys_name, exec_k, exact_k, sv_k in [
    ("Base model",         "base_exec_acc",    "base_exact_acc",    "mean_base_sv"),
    ("+ LoRA (M1)",        "tuned_exec_acc",   "tuned_exact_acc",   "mean_tuned_sv"),
    ("+ Reranker (M2)",    "rerank_exec_acc",  "rerank_exact_acc",  "mean_rerank_sv"),
    ("+ Hybrid (M3)",      "hybrid_exec_acc",  "hybrid_exact_acc",  "mean_hybrid_sv"),
]:
    print(f"  {sys_name:<25} {summary[exec_k]:>9.1%} "
          f"{summary[exact_k]:>9.1%} {summary[sv_k]:>10.4f}")

Saved full eval: /content/drive/MyDrive/CS_512_Project/milestone3_full_eval/csv_outputs/milestone3_full_eval.csv
Saved summary:   /content/drive/MyDrive/CS_512_Project/milestone3_full_eval/csv_outputs/milestone3_summary.json

Milestone 3 Final Summary:
  System                      Exec Acc  Exact Acc    Mean SV
  ---------------------------------------------------------
  Base model                    15.6%      0.7%     0.2335
  + LoRA (M1)                   13.8%      8.7%     0.4364
  + Reranker (M2)               20.7%     13.1%     0.6421
  + Hybrid (M3)                 20.7%     13.1%     0.6421
